## Remote ID Sppoofing Attack in a City

In [ ]:
import pickle
from typing import Literal

from simulator import Simulator
from simulator.config import DATA_PATH, Color
from simulator.entities import SimVehicle
from simulator.helpers import SimProcess, clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.planner import AutoPlan
from simulator.visualizer import Gazebo, GazMarker

clean()

## Simulation Positions

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

base_homes = ENUPose.list([(15, -10, 0, 0), (0, -15, 0, 0)])
base_paths = [ENU.list([(0, 0, 5), (0, 25, 5)]), ENU.list([(0, 0, 5), (30, 0, 5)])]

## Create Vehicles

In [ ]:
sysids = [1, 255]  # attacker sysid is (temporally) being hardcoded to 255

colors = [Color.GREEN, Color.RED]
speeds = [3.0, 3.0]  # m/s

gcs_name = f"Multicolor_{''.join([color.emoji for color in colors])}"
mission_folder = DATA_PATH / "missions"
mission_folder.mkdir(parents=True, exist_ok=True)

vehs: list[SimVehicle] = []
for sysid, base_home, base_path, color, speed in zip(
    sysids, base_homes, base_paths, colors, speeds, strict=True
):
    mission_path = str(mission_folder / f"mission_{sysid}.waypoints")
    auto_plan = AutoPlan.from_relative_path(
        name="simple_auto_plan",
        sysid=sysid,
        gra_origin=gra_origin,
        relative_home=base_home,
        relative_path=base_path,
        navigation_speed=speed,
        mission_path=mission_path,
    )

    veh = SimVehicle.from_relative(
        sysid=sysid,
        gcs_name=f"{color.name}_{color.emoji}",
        plan=auto_plan,
        color=color,
        enu_origin=enu_origin,
        relative_home=base_home,
        relative_path=base_path,
        model="gazebo-iris",
    )
    vehs.append(veh)

## Scenario configurarion

In [ ]:
fake_pos = ENU(x=15, y=0, z=5)
avoidance_method: Literal["stop", "naive"] = "stop"

radar_radius = 10  # meters
safety_radius = 5  # meters

# This is temporary to visualize the scenario
with open(DATA_PATH / "fake_position.pkl", "wb") as f:
    pickle.dump(fake_pos, f)

## Gazebo

In [ ]:
small_city_path = "simulator/visualizer/gazebo/worlds/small_city_demo.world"
gaz = Gazebo(gra_origin, world_path=small_city_path)

origin_gaz = GazMarker(
    name="origin", group="origin", pos=enu_origin.unpose(), color=Color.WHITE
)

fake_marker = GazMarker(
    name="fake_pos", group="fake_pos", pos=fake_pos, color=Color.ORANGE
)

avoid_zone = GazMarker(
    name="avoid_zone",
    group="avoid_zone",
    pos=fake_pos,
    color=Color.RED,
    radius=safety_radius,
    alpha=0.85,
)

for marker in [origin_gaz, fake_marker, avoid_zone]:
    gaz.markers.append(marker)

## Simulator

In [ ]:
simulator = Simulator(
    visualizer=gaz,
    terminals=[SimProcess.GCS],
    verbose=1,
)
for veh in vehs:
    simulator.add_vehicle(veh)

simulator.show()

## Run

In [ ]:
orac = simulator.launch()
orac.run()